# Imports

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

# Read the data

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/ziad23samer/cleaned-brainrot/cleaned.csv")

In [ ]:
X = df.drop(columns=["brain_rot_index"]).values
y = df["brain_rot_index"].values

In [ ]:
print(X.shape)

(468671, 29)


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
print(X_train.shape, X_val.shape, X_test.shape)

(281202, 29) (93734, 29) (93735, 29)


# Models

## Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)

LinearRegression()

In [ ]:
y_pred = baseline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse}")

MSE: 20.858028823045217


## Advanced Imports

In [ ]:
from sklearn.model_selection import (
    RepeatedKFold, GridSearchCV, cross_val_score
)

In [ ]:
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

In [ ]:
def get_cv_scores(model, X, y):
    scores = cross_val_score(
        model, X, y,
        cv=rkf, scoring="r2", n_jobs=-1
    )
    return scores

## Linear Regression

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
param_grid = {
    "alpha": [0.1, 1, 10]
}

In [ ]:
ridge = Ridge()

In [ ]:
scores = get_cv_scores(ridge, X_train, y_train)

In [ ]:
print(scores)

[0.71027126 0.71057129 0.71142486 0.70954272 0.70583087 0.70941289
 0.70791921 0.70823753 0.71128022 0.71075997 0.70972083 0.70901957
 0.71118669 0.70974067 0.70796158]


In [ ]:
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

In [ ]:
grid_search = GridSearchCV(
        estimator  = ridge,
        param_grid = param_grid,
        cv         = rkf,
        scoring    = "r2",
        n_jobs     = -1,
        verbose    = 0,
        refit      = True,
)
grid_search.fit(X_trainval, y_trainval)

best_LR = grid_search.best_estimator_
print({
    "best_params": grid_search.best_params_,
    "best_cv_r2" : grid_search.best_score_,
})

{'best_params': {'alpha': 10}, 'best_cv_r2': np.float64(0.7095124955820695)}


In [ ]:
y_pred = best_LR.predict(X_test)

In [ ]:
mse = mean_squared_error(y_pred, y_test)
print(f"Mean Squared Error of the linear regression is {mse}")

Mean Squared Error of the linear regression is 20.856728341553463


## AdaBoost Regressor

In [ ]:
from sklearn.ensemble import AdaBoostRegressor

In [ ]:
param_grid = {
    "n_estimators":  [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.5, 1.0],
    "loss":          ["linear", "square", "exponential"],
}

In [ ]:
ada = AdaBoostRegressor()

In [ ]:
scores = get_cv_scores(ada, X_train, y_train)

In [ ]:
print(scores)

[0.66206675 0.65974755 0.66211955 0.65944932 0.65548344 0.66115048
 0.65909497 0.65538641 0.6578809  0.66078096 0.65805635 0.65654979
 0.65960051 0.66168635 0.65679574]


In [ ]:
grid_search = GridSearchCV(
        estimator  = ada,
        param_grid = param_grid,
        cv         = rkf,
        scoring    = "r2",
        n_jobs     = -1,
        verbose    = 0,
        refit      = True,
)
grid_search.fit(X_trainval, y_trainval)

best_LR = grid_search.best_estimator_
print({
    "best_params": grid_search.best_params_,
    "best_cv_r2" : grid_search.best_score_,
})

In [ ]:
y_pred = best_LR.predict(X_test)
mse = mean_squared_error(y_pred, y_test)
print(f"Mean Squared Error of the linear regression is {mse}")

## SVM

In [ ]:
from sklearn.svm import SVR

In [ ]:
from sklearn.linear_model import SGDRegressor
svm = SGDRegressor(loss='epsilon_insensitive')


In [ ]:
scores = get_cv_scores(svm, X_train, y_train)
print(scores)

[0.70940685 0.71046289 0.71099914 0.70845428 0.7054184  0.709273
 0.70745482 0.70796474 0.71097444 0.71059707 0.70946866 0.70860331
 0.71099598 0.70949755 0.70758807]


In [ ]:
"""
SGDRegressor(loss='squared_error', *, penalty='l2',
alpha=0.0001, l1_ratio=0.15, fit_intercept=True,
max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1,
random_state=None, learning_rate='invscaling', eta0=0.01, power_t=0.25,
early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, warm_start=False, average=False)
"""

In [ ]:
param_grid = {
    "C":       [0.1, 1, 10, 100],
    "epsilon": [0.01, 0.1, 0.5],
    "kernel":  ["rbf", "linear"],
    "gamma":   ["scale", "auto"]
}

In [ ]:
grid_search = GridSearchCV(
        estimator  = svm,
        param_grid = param_grid,
        cv         = rkf,
        scoring    = "r2",
        n_jobs     = -1,
        verbose    = 0,
        refit      = True,
)
grid_search.fit(X_trainval, y_trainval)

best_LR = grid_search.best_estimator_
print({
    "best_params": grid_search.best_params_,
    "best_cv_r2" : grid_search.best_score_,
})